# KeyFlow Pipeline Verification & Comparison (Google Colab)

> **Step 1: Enable GPU Runtime**
>
> Before running any cells, ensure a GPU accelerator is connected:
> Navigate to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** (or any available GPU), then click **Save**.

In [2]:
# Confirm GPU availability
!nvidia-smi

Tue Nov 10 16:01:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# @title Clone KeyFlow Repository
# @markdown Enter the Git URL of your fork or repo. Note: edit `<user>` to match your GitHub account/repo.
repo_url = "https://github.com/luizccc-cpu/Piano-Flow.git"  # @param {type:"string"}

import os

repo_dir = "/content/Piano-Flow"
if os.path.exists(repo_dir):
    print(f"Target directory {repo_dir} already exists. Pulling latest commits...")
    !cd {repo_dir} && git pull
else:
    print(f"Cloning {repo_url} into {repo_dir}...")
    !git clone {repo_url} {repo_dir}

%cd {repo_dir}

Target directory /content/Piano-Flow already exists. Pulling latest commits...
Already up to date.
/content/Piano-Flow


In [4]:
# @title Alternative: Upload backend/ folder as a ZIP
# @markdown If you prefer not to clone from GitHub (e.g. uncommitted local changes), zip your local `backend/` folder and upload it here.
import os
import zipfile
from google.colab import files

print("Optional: Upload backend.zip (skip this cell if you cloned the repo in the previous cell)...")
uploaded_zips = files.upload()

for fname in uploaded_zips.keys():
    if fname.lower().endswith(".zip"):
        extract_target = "/content/Piano-Flow"
        os.makedirs(extract_target, exist_ok=True)
        with zipfile.ZipFile(fname, "r") as zf:
            zf.extractall(extract_target)
        print(f"Successfully extracted {fname} into {extract_target}!")
        %cd {extract_target}
        break

Optional: Upload backend.zip (skip this cell if you cloned the repo in the previous cell)...


In [5]:
# Install system audio dependencies and Python requirements
!apt-get update -qq && apt-get -y install ffmpeg libsndfile1
!pip install -r backend/requirements.txt

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:6.1.1-3ubuntu5).
libsndfile1 is already the newest version (1.2.2-1ubuntu5.24.04.1).
0 upgraded, 0 newly installed, 0 to remove and 86 not upgraded.


In [6]:
# Check installed dependency versions to verify compatibility
import importlib

check_packages = [
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("torchaudio", "torchaudio"),
    ("audio-separator", "audio_separator"),
    ("transkun", "transkun"),
    ("demucs", "demucs"),
    ("piano_transcription_inference", "piano_transcription_inference"),
]

print("=== Installed Dependency Versions ===")
for display_name, mod_name in check_packages:
    try:
        m = importlib.import_module(mod_name)
        v = getattr(m, "__version__", "installed (no __version__)")
        print(f"  {display_name:<30} : {v}")
    except ImportError as err:
        print(f"  {display_name:<30} : NOT FOUND ({err})")

=== Installed Dependency Versions ===
  numpy                          : 1.26.4
  torch                          : 2.14.1+cu130
  torchaudio                     : 2.11.0+cu130
  audio-separator                : installed (no __version__)
  transkun                       : installed (no __version__)
  demucs                         : 4.1.0
  piano_transcription_inference  : installed (no __version__)


In [7]:
# Upload a test song audio file
from google.colab import files
import os

print("Upload an audio file (e.g., .mp3, .wav, .m4a, .flac) to run through the pipelines:")
uploaded_audio = files.upload()

if not uploaded_audio:
    raise ValueError("No file uploaded. Please run this cell again and select an audio file.")

test_audio_path = list(uploaded_audio.keys())[0]
print(f"Ready: {test_audio_path} ({os.path.getsize(test_audio_path)} bytes)")

Upload an audio file (e.g., .mp3, .wav, .m4a, .flac) to run through the pipelines:


Saving Damien Rice - 9 Crimes (Instrumental).mp3 to Damien Rice - 9 Crimes (Instrumental).mp3
Ready: Damien Rice - 9 Crimes (Instrumental).mp3 (5211117 bytes)


In [8]:
# Run compare_pipelines.py across all pipeline combinations
# (Add --skip-demucs to test only the BS-RoFormer / Transkun pipeline)
!python backend/tools/compare_pipelines.py "{test_audio_path}"

KeyFlow Pipeline Comparison Tool
Input Audio  : Damien Rice - 9 Crimes (Instrumental).mp3
Output Dir   : Damien Rice - 9 Crimes (Instrumental)_compare
Sensitivity  : 55.0
Skip Demucs  : False

[1/4] Running HT-Demucs 6S separation (once for Rows A & B)...
  -> Saved raw HT-Demucs stem in 20.07s: htdemucs_6s_raw.wav
  -> Saved legacy prepped stem in 31.47s: htdemucs_6s_legacy_prep.wav
  -> Saved minimal prepped stem in 0.65s: htdemucs_6s_minimal_prep.wav

[2/4] Running BS-RoFormer SW separation (once for Rows C & D)...
2026-11-10 16:03:45,495 - INFO - separator - Separator version 0.30.2 instantiating with output_dir: Damien Rice - 9 Crimes (Instrumental)_compare/sep_out, output_format: WAV
2026-11-10 16:03:45,495 - INFO - separator - Using model directory from model_file_dir parameter: /tmp/audio-separator-models
2026-11-10 16:03:45,495 - INFO - separator - Operating System: Linux #1 SMP Thu Oct 22 03:07:54 UTC 2026
2026-11-10 16:03:45,496 - INFO - separator - System: Linux Node: 65cb0

In [9]:
# Zip and download the comparison output folder
from pathlib import Path
import shutil
from google.colab import files

stem = Path(test_audio_path).stem
output_dir = Path(f"{stem}_compare")
zip_name = f"{stem}_compare.zip"

if output_dir.exists():
    print(f"Archiving {output_dir} -> {zip_name}...")
    shutil.make_archive(f"{stem}_compare", "zip", output_dir)
    print("Downloading comparison zip archive...")
    files.download(zip_name)
else:
    print(f"Warning: Output directory {output_dir} not found. Check previous cell output for errors.")

Archiving Damien Rice - 9 Crimes (Instrumental)_compare -> Damien Rice - 9 Crimes (Instrumental)_compare.zip...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Start uvicorn server in background, probe /health, and test POST /api/transcribe
import subprocess
import time
import requests
import json
import os
from pathlib import Path

# Ensure port 8000 is clean
!fuser -k 8000/tcp || true

backend_dir = Path("/content/Piano-Flow/backend")
if not backend_dir.exists():
    backend_dir = Path("backend").resolve()

print("Starting uvicorn server:app on http://127.0.0.1:8000 in the background...")
server_proc = subprocess.Popen(
    ["uvicorn", "server:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd=str(backend_dir),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

health_data = None
print("Probing /health...")
for attempt in range(20):
    time.sleep(1)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2)
        if r.status_code == 200:
            health_data = r.json()
            print("Server /health is UP:")
            print(json.dumps(health_data, indent=2))
            break
    except Exception:
        pass

if not health_data:
    server_proc.terminate()
    out, _ = server_proc.communicate(timeout=5)
    raise RuntimeError(f"Server failed to start. Logs:\n{out}")

print(f"\nPOSTing '{test_audio_path}' to http://127.0.0.1:8000/api/transcribe ...")
try:
    with open(test_audio_path, "rb") as f:
        files_payload = {"file": (Path(test_audio_path).name, f, "audio/mpeg")}
        form_payload = {"isolate_piano": "true", "sensitivity": "0.5"}
        resp = requests.post(
            "http://127.0.0.1:8000/api/transcribe",
            files=files_payload,
            data=form_payload,
            timeout=300,
        )

    if resp.status_code == 200:
        res = resp.json()
        print("\n=== Transcription Succeeded ===")
        print(f"Title:            {res.get('title')}")
        print(f"Separator used:   {res.get('separator_used')}")
        print(f"Transcriber used: {res.get('transcriber_used')}")
        print(f"Timings:          {json.dumps(res.get('timings'), indent=2)}")
        print(f"MIDI Base64 len:  {len(res.get('midi_base64', ''))} chars")
    else:
        print(f"Transcription failed ({resp.status_code}): {resp.text}")
finally:
    print("\nStopping uvicorn server...")
    server_proc.terminate()
    try:
        server_proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        server_proc.kill()
    print("Server stopped.")

Starting uvicorn server:app on http://127.0.0.1:8000 in the background...
Probing /health...
Server /health is UP:
{
  "status": "ok",
  "version": "3.1.0",
  "engines": [
    "BS-RoFormer",
    "HT-Demucs-6S",
    "Transkun",
    "ByteDance-Piano",
    "Librosa-CQT",
    "Sheet-OMR"
  ],
  "audio_separator_available": true,
  "demucs_available": true,
  "transkun_available": true,
  "bytedance_available": true,
  "basic_pitch_available": false,
  "librosa_available": true,
  "pdfium_available": true,
  "omr_available": true
}

POSTing 'Damien Rice - 9 Crimes (Instrumental).mp3' to http://127.0.0.1:8000/api/transcribe ...

=== Transcription Succeeded ===
Title:            Damien Rice - 9 Crimes (Instrumental)
Separator used:   htdemucs_6s
Transcriber used: transkun
Timings:          {
  "separation_s": 18.91,
  "transcription_s": 25.17
}
MIDI Base64 len:  3032 chars

Stopping uvicorn server...
Server stopped.


In [1]:
import glob, logging, traceback
from audio_separator.separator import Separator

song = glob.glob("/content/*.mp3")[0]
print("Song:", song)
try:
    sep = Separator(output_dir="/content/roformer_test",
                    model_file_dir="/tmp/audio-separator-models",
                    output_format="WAV", log_level=logging.INFO)
    sep.load_model(model_filename="BS-Roformer-SW.ckpt")
    print("OUTPUT FILES:", sep.separate(song))
except Exception:
    traceback.print_exc()

ModuleNotFoundError: No module named 'audio_separator'

In [2]:
!audio-separator --list_models 2>/dev/null | grep -i -E "piano|roformer-sw|BS-Roformer-SW"